In [1]:
"""
The purpose of this Juyter notebook is to format the training and
validation data such that it can be directly passed to the training
script.
"""

'\nThe purpose of this Juyter notebook is to format the training and\nvalidation data such that it can be directly passed to the training\nscript.\n'

In [2]:
import os
import ast

import numpy as np
import pandas as pd

# Create Subdirectories

In [ ]:
# Create a subdirectory to store the formatted data in
formatted_data_dir = "formatted_dataset"

if not os.path.exists(formatted_data_dir):
    os.makedirs(formatted_data_dir)

# Data Formatting

## Loading Data

In [ ]:
# Load the training set
training_set_path = (
    "dataset_prior_to_formatting/PU_training_set.tsv"
)

training_set_df = pd.read_csv(
    training_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the validation set
val_set_path = (
    "dataset_prior_to_formatting/PU_validation_set.tsv"
)

val_set_df = pd.read_csv(
    val_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the test set
test_set_path = (
    "dataset_prior_to_formatting/PU_test_set.tsv"
)

test_set_df = pd.read_csv(
    test_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the PPI probability data
ppi_preds_path = (
    "dataset_prior_to_formatting/all_ppi_predictions.tsv"
)

ppi_preds_df = pd.read_csv(
    ppi_preds_path,
    sep="\t"
)

## Formatting the Data

In [ ]:
# For the sake of convenience, define a function performing data
# formatting
def format_data(data_set_df, ppi_df, ppi_feature="interaction_probability"):
    """
    Format training and validation data such that it can be fed into the
    PU learning architecture.

    Parameters
    ----------
    data_set_df: Pandas DataFrame
        The unformatted data set to be subjected to formatting.
    ppi_df: Pandas DataFrame
        A Pandas DataFrame storing the interactions probabilities used
        in PU learning.
    ppi_feature: str, default="interaction_probability"
        A string indicating whether to use probabilities or logits for
        the `ppi_vec` feature. Valid options are
        "interaction_probability" and "logits".
    """
    # Create a dictionary mapping UniProt accessions to their
    # corresponding PPI probabiity vector of length 440 (or PPI logit
    # vector)
    prot_to_ppi_prob_vector_dict = {}

    # To this end, the `.groupby()` method is employed
    for protein, group in ppi_df.groupby(by="protein_1"):
        ppi_prob_vector = group[ppi_feature].to_numpy()
        assert ppi_prob_vector.shape[0] == 440, (
            f"For protein {protein}, some PPI probabilities are missing!"
        )
        prot_to_ppi_prob_vector_dict[protein] = ppi_prob_vector

    # The probabilities must first be aggregated as some genes encode
    # multiple proteins
    # Create a dictionary mapping gene names to the proteins they encode
    # Conveniently enough, multiple occurrences of individual genes have
    # already been collaped and the proteins encoded by the individual
    # genes are stored as lists in the `protein_ids` column
    gene_to_prot_dict = (
        data_set_df.set_index("gene")["protein_ids"].to_dict()
    )

    # For each gene, aggregate the corresponding PPI probability vectors
    agg_ppi_prob_vector_dict = {}

    for gene, prot_list in gene_to_prot_dict.items():
        ppi_prob_vector_list = [
            prot_to_ppi_prob_vector_dict[protein]
            for protein in prot_list
        ]
        agg_ppi_prob_vector_dict[gene] = np.max(
            ppi_prob_vector_list, axis=0
        ).tolist()
    
    formatted_df = data_set_df.copy()
    
    # Add a column to the `data_set_df` DataFrame storing the aggregated
    # PPI probability vectors
    # The aggregated vectors have to be arranged in the correct order
    formatted_df["ppi_vec"] = data_set_df["gene"].apply(
        lambda gene: agg_ppi_prob_vector_dict[gene]
    )

    # Retain and reorder columns of interest
    formatted_df = formatted_df[[
        "gene", "phenotype_vec", "ppi_vec", "label"
    ]]

    return formatted_df

In [6]:
# Format the training, validation and test set using probabilities as
# PPI feature and save them to disk
formatted_training_df = format_data(
    training_set_df, ppi_preds_df
)

formatted_training_df.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_training_set_formatted_with_probs.tsv"
    ),
    sep="\t",
    index=False
)

formatted_val_df = format_data(
    val_set_df, ppi_preds_df
)

formatted_val_df.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_val_set_formatted_with_probs.tsv"
    ),
    sep="\t",
    index=False
)

formatted_test_df = format_data(
    test_set_df, ppi_preds_df
)

formatted_test_df.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_test_set_formatted_with_probs.tsv"
    ),
    sep="\t",
    index=False
)

## Row Permutation of PPI Vectors

In [ ]:
# As an ablation study, a row permutation of PPI vectors is performed,
# i.e. the PPI vectors remain intact are re-assigned to different genes
# so as to destroy the correspondence between human genes and their
# interaction profiles
formatted_training_df = pd.read_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_training_set_formatted_with_probs.tsv"
    ),
    sep="\t",
    converters={
        "ppi_vec": ast.literal_eval
    }
)

formatted_train_df_row_permutation = formatted_training_df.copy()

formatted_train_df_row_permutation["ppi_vec"] = (
    formatted_train_df_row_permutation["ppi_vec"]
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

formatted_train_df_row_permutation.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_training_set_formatted_with_probs_pu_ratio_1_5_ppi_"
        "vec_row_permutation.tsv"
    ),
    sep="\t",
    index=False
)

## Column Permutation of PPI Vectors

In [ ]:
# As another ablation study, a column permutation of PPI vectors is
# performed, i.e. the PPI vectors are destroyed by permuting the
# interaction probabilities across human proteins
# This destroys the original interaction signatures
formatted_training_df = pd.read_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_training_set_formatted_with_probs.tsv"
    ),
    sep="\t",
    converters={
        "ppi_vec": ast.literal_eval
    }
)

formatted_train_df_col_permutation = formatted_training_df.copy()

# Expand the lists into columns
ppi_vecs_df = pd.DataFrame(
    formatted_train_df_col_permutation["ppi_vec"].tolist()
)

# Independently shuffle each column
rng = np.random.default_rng()

for col in ppi_vecs_df.columns:
    ppi_vecs_df[col] = rng.permutation(ppi_vecs_df[col])

# Collapse columns back into lists
formatted_train_df_col_permutation["ppi_vec"] = (
    ppi_vecs_df.to_numpy().tolist()
)

# Save the DataFrame with column-wise shuffled PPI vectors to disk
formatted_train_df_col_permutation.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_training_set_formatted_with_probs_pu_ratio_1_5_ppi_"
        "vec_col_permutation.tsv"
    ),
    sep="\t",
    index=False
)

# Data Formatting for the Entire Screen

In [ ]:
# Load the TSV file with phenotype vectors and labels (amongst others)
# for the entire screen
entire_screen_path = (
    "dataset_prior_to_formatting/DataFrame_for_train_validation_test_"
    "split.tsv"
)

entire_screen_df = pd.read_csv(
    entire_screen_path,
    sep="\t",
    converters={"phenotype_vec": ast.literal_eval}
)

In [6]:
# Prior to the actual formatting, some pre-formatting has to be applied
# This pre-formatting consists of collapsing the entries/rows to the
# gene level, i.e. making each gene occupy only one row
# To this end, the `protein_id` column is replaced with a `protein_ids`
# column inside which all proteins encoded by a specific gene are stored
# in a list

# Group the DataFrame by the gene names, extract the corresponding
# protein IDs and add a list of them to the new `protein_ids` column
entire_screen_df = (
    entire_screen_df
    .groupby("gene", as_index=False)
    .agg({
        "protein_id": list,
        "phenotype_vec": "first",
        "label": "first"
    })
    .rename(columns={"protein_id": "protein_ids"})
)

In [ ]:
# Now that pre-formatting has been performed, apply the actual
# formatting
entire_screen_formatted_df = format_data(
    entire_screen_df,
    ppi_preds_df
)

entire_screen_formatted_df.to_csv(
    os.path.join(
        formatted_data_dir,
        "all_hf_entire_screen_formatted_with_probs.tsv"
    ),
    sep="\t",
    index=False
)